# Week 14 — Project Guide and Template

**MCTA 4364 Machine Vision** · Group project · demonstrations in Week 14

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W14_project/W14_project_guide.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W14_project/W14_project_guide.ipynb)

The project is where the threads of the course meet: you choose a real vision problem, collect or build the data, design a pipeline (classical,
learned, or — usually best — both), **evaluate it honestly**, and demonstrate it live. This notebook is your guide *and* your template: Section 3
runs a complete miniature project end to end so you can copy its structure.

> Your instructor's official brief, dates and marking scheme take precedence over anything in this notebook.

### What you will be able to do
1. Turn a vague idea into a measurable machine-vision problem with a clear success criterion.
2. Plan data collection, labelling and splits that give an honest estimate of performance.
3. Build a **baseline first**, then improve it using error analysis.
4. Report accuracy, **per-class** errors, **latency** and **robustness**, and demonstrate the system live.

### Milestones
| Week | Milestone | What to hand in |
|---|---|---|
| 3 | Groups formed | group members, a first idea |
| 5 | **Proposal** | problem, success criterion, data plan, baseline idea, risks (template in Section 5) |
| 9 | **Dataset and baseline** | labelled data with splits, a working baseline and its numbers |
| 12 | **Progress check** | improved pipeline, error analysis, what is left |
| 14 | **Demonstration** | live demo, results, short report and code |

## 0. Setup

In [ ]:
# Setup: works in Google Colab, Kaggle, Jupyter/VS Code on your laptop, and CI.
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git"
if not os.path.exists("resources/scripts/cvhelpers.py"):
    if os.path.exists("../../resources/scripts/cvhelpers.py"):  # opened from its own folder
        os.chdir("../..")
    else:  # Colab / Kaggle: fetch the course helpers
        if not os.path.isdir("MCTA-4364-Machine-Vision"):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
        os.chdir("MCTA-4364-Machine-Vision")
print("Working directory:", os.getcwd())

# Install only what is missing. PyTorch is deliberately NOT pip-installed here:
# that would replace your laptop's CUDA build with a CPU-only one (see SETUP.md).
needed = {"cv2": "opencv-python", "skimage": "scikit-image", "ipywidgets": "ipywidgets",
          "matplotlib": "matplotlib", "scipy": "scipy", "plotly": "plotly"}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
sys.path.append("resources/scripts")

In [ ]:
import time

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

import partsim
from cvhelpers import SMOKE, check, interact, pipeline_diagram, quiz

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11})
NAMES = partsim.SCENE_CLASSES

## 1. Choosing a problem

A good project problem is **specific**, **measurable** and **demonstrable in a room** with a laptop and a camera. Write it as one sentence:

> *"Count and classify the four fastener types on a moving tray at ≥ 10 frames/s on a laptop, with ≥ 95 % recall for every type under office lighting."*

### Idea bank (mapped to the course)
| idea | core weeks | a measurable target |
|---|---|---|
| coin / fastener counting and sorting | W04, W08–W09 | count error per image, per-class recall |
| dimensional gauge (diameter, hole spacing) with a calibrated camera | W02, W04, W07A | measurement error in mm vs a caliper |
| label / date-code reading and verification | W03, W13A | character or string accuracy |
| surface defect inspection (fabric, PCB, printed sheets) | W09A, W11B | recall at a fixed false-alarm rate |
| people / vehicle counting and speed from video | W10 | count error, speed error vs ground truth |
| object tracking for a pick-and-place robot | W10B, W12A | position error, track switches |
| 3D measurement of box sizes with a stereo or depth camera | W12 | volume error |
| panorama / document scanning app | W05B, W03A | reprojection error, OCR accuracy |
| PPE (helmet / vest) compliance at a workstation | W08–W09, W13A | recall on unseen people, false alarms per hour |
| auto-labelled dataset + detector for a new object class | W13A, W09B | labelling time saved, mAP vs manual labels |

### Data sources
- **Your own images and video** (best: it is *your* problem). Ask permission before filming people; do not publish faces without consent.
- **Open datasets** — check the licence first: VisA (CC BY 4.0), COCO (CC BY 4.0 annotations; image licences vary), Open Images, Roboflow Universe
  (per-dataset licences). MVTec AD is **CC BY-NC-SA** and must not be redistributed.
- **Synthetic data** (W13B) to supplement, never to replace, a **real test set**.

## 2. The workflow that works

1. **Define the metric and the target** before writing code (what counts as a success? per image, per object, per frame?).
2. **Split the data by situation**, not by image: different days, parts, people or videos in train / validation / test (Week 9A leakage).
3. **Baseline first** — the simplest thing that could work (thresholds and contours, a pretrained detector, zero-shot CLIP). Measure it.
4. **Error analysis**: look at the failures, group them, fix the biggest group (more/better data, a better method, better lighting!).
5. **Iterate**, changing one thing at a time, and keep a results table.
6. **Test once** on the untouched test set, then measure latency and robustness, and prepare the demo.

In [ ]:
fig, ax = plt.subplots(figsize=(22, 2.6))
pipeline_diagram(["problem +\nmetric + target", "data plan\n(splits by situation)", "baseline", "error analysis", "improve\n(data / method / optics)",
                  "final test\n(once)", "latency +\nrobustness", "demo + report"],
                 colors=["#fef7e0", "#e8f0fe", "#e6f4ea", "#fce8e6", "#e6f4ea", "#fce8e6", "#e8f0fe", "#fef7e0"], ax=ax, fontsize=10)
ax.set_title("Figure 2.1 - The project loop: measure, look at the errors, improve, repeat", fontweight="bold")
plt.show()

## 3. A worked mini-project (copy this structure)

**Problem.** *Count and classify hex nuts, washers, bolts and flanges on a conveyor image; target ≥ 95 % recall for every class and ≥ 20 images/s
on a laptop CPU.* Data come from `partsim` so that this template runs anywhere; in your project, replace them with your own images.

### 3.1 Data and splits
Training/validation and test scenes use different random seeds (different "days"); a second test set uses a darker, noisier camera.

In [ ]:
N_DEV, N_TEST = (8, 8) if SMOKE else (40, 60)
dev_rng, test_rng, dark_rng = np.random.default_rng(1), np.random.default_rng(2), np.random.default_rng(3)
dev = [partsim.make_scene(dev_rng, 5) for _ in range(N_DEV)]                    # for designing and tuning
test = [partsim.make_scene(test_rng, 5) for _ in range(N_TEST)]                 # touched once, at the end
test_dark = [partsim.make_scene(dark_rng, 5, exposure=0.6, noise=6.0) for _ in range(N_TEST)]
counts = np.bincount(np.concatenate([l for _, _, l in test]), minlength=4)
print(f"{N_DEV} development scenes, {N_TEST} test scenes, {N_TEST} dark-camera scenes | test objects per class:",
      dict(zip(NAMES, counts.tolist())))

### 3.2 Baseline: threshold, contours and shape rules (Weeks 3–4)
Segment with Otsu's threshold, take the outer contours, and classify each blob from its **holes**, **circularity** and **area**.

In [ ]:
def blobs(img, thresh=None):
    g = cv2.GaussianBlur(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    if thresh is None:
        _, m = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    else:
        m = (g > thresh).astype(np.uint8) * 255
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    cs, hier = cv2.findContours(m, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
    out = []
    for i, c in enumerate(cs):
        if hier[0][i][3] != -1 or cv2.contourArea(c) < 80:           # outer contours of a reasonable size only
            continue
        holes, j = 0, hier[0][i][2]
        while j != -1:
            holes += cv2.contourArea(cs[j]) > 8
            j = hier[0][j][0]
        area, per = cv2.contourArea(c), cv2.arcLength(c, True)
        x, y, w, h = cv2.boundingRect(c)
        out.append(dict(box=np.array([x, y, x + w, y + h], float), area=area, holes=holes, circ=4 * np.pi * area / per ** 2))
    return out


def classify(b):
    if b["holes"] >= 3:
        return NAMES.index("flange")
    if b["circ"] < 0.55:
        return NAMES.index("bolt")
    return NAMES.index("hex_nut") if b["area"] < 1250 else NAMES.index("washer")


def pipeline_v1(img):
    bs = blobs(img)
    return np.array([b["box"] for b in bs]).reshape(-1, 4), np.array([classify(b) for b in bs], int)


def box_iou(a, b):
    a, b = np.asarray(a, float).reshape(-1, 4), np.asarray(b, float).reshape(-1, 4)
    x0 = np.maximum(a[:, None, 0], b[None, :, 0]); y0 = np.maximum(a[:, None, 1], b[None, :, 1])
    x1 = np.minimum(a[:, None, 2], b[None, :, 2]); y1 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x1 - x0, 0, None) * np.clip(y1 - y0, 0, None)
    area = lambda z: (z[:, 2] - z[:, 0]) * (z[:, 3] - z[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)


def evaluate(pipeline, scenes, iou_thr=0.5):
    """Object-level evaluation: per-class TP / FP / FN with greedy IoU matching, plus the count error per image."""
    tp, fp, fn = np.zeros(4, int), np.zeros(4, int), np.zeros(4, int)
    count_err = []
    for img, gb, gl in scenes:
        pb, pl = pipeline(img)
        iou = box_iou(pb, gb)
        used = set()
        for i in range(len(pb)):
            cand = [j for j in range(len(gb)) if j not in used and gl[j] == pl[i] and iou[i, j] >= iou_thr]
            if cand:
                used.add(max(cand, key=lambda j: iou[i, j])); tp[pl[i]] += 1
            else:
                fp[pl[i]] += 1
        for j in range(len(gb)):
            fn[gl[j]] += j not in used
        count_err.append(abs(len(pb) - len(gb)))
    return dict(precision=tp / np.maximum(tp + fp, 1), recall=tp / np.maximum(tp + fn, 1), mean_count_error=float(np.mean(count_err)))


res_v1 = evaluate(pipeline_v1, dev)
print("baseline on the DEVELOPMENT set")
for k, n in enumerate(NAMES):
    print(f"  {n:<8} precision {res_v1['precision'][k]:.2f} recall {res_v1['recall'][k]:.2f}")
print(f"  mean |count error| per image: {res_v1['mean_count_error']:.2f}")

### 3.3 Error analysis: look at the failures
Numbers say *that* bolts fail; pictures say *why*. Always build an error gallery.

In [ ]:
def draw(img, pipeline):
    pb, pl = pipeline(img)
    return partsim.draw_boxes(img, pb, pl)


worst = sorted(range(len(dev)), key=lambda i: -abs(len(pipeline_v1(dev[i][0])[0]) - len(dev[i][1])))[:3]
fig, axes = plt.subplots(2, 3, figsize=(24, 11))
for k, i in enumerate(worst):
    img, gb, gl = dev[i]
    axes[0, k].imshow(draw(img, pipeline_v1)[..., ::-1]); axes[0, k].set_title(f"baseline: {len(pipeline_v1(img)[0])} objects found")
    g = cv2.GaussianBlur(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    otsu, _ = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    axes[1, k].hist(g.ravel(), 128, color="#5f6368"); axes[1, k].axvline(otsu, color="#d93025", lw=2, label=f"Otsu threshold {otsu:.0f}")
    axes[1, k].set_yscale("log"); axes[1, k].set_xlabel("grey level"); axes[1, k].legend()
    axes[1, k].set_title(f"ground truth: {len(gb)} objects ({sum(np.array(gl) == 2)} bolts)")
for ax in axes[0]:
    ax.axis("off")
fig.suptitle("Figure 3.1 - Error gallery: the dark bolts fall below Otsu's threshold, which is pulled up by the bright parts", fontweight="bold")
plt.tight_layout(); plt.show()

### 3.4 Improvement: a threshold relative to the belt
The belt is the most common grey level, so a threshold a fixed step **above the belt** keeps the dark bolts. One change, one new measurement.

In [ ]:
def belt_threshold(img, margin=18):
    g = cv2.GaussianBlur(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    return float(np.argmax(np.bincount(g.ravel(), minlength=256))) + margin


def pipeline_v2(img, margin=18):
    bs = blobs(img, belt_threshold(img, margin))
    return np.array([b["box"] for b in bs]).reshape(-1, 4), np.array([classify(b) for b in bs], int)


res_v2 = evaluate(pipeline_v2, dev)
fig, ax = plt.subplots(figsize=(12, 4.5))
xs = np.arange(4)
ax.bar(xs - 0.2, res_v1["recall"], 0.4, label=f"v1 Otsu (count error {res_v1['mean_count_error']:.2f})", color="#9aa0a6")
ax.bar(xs + 0.2, res_v2["recall"], 0.4, label=f"v2 belt-relative (count error {res_v2['mean_count_error']:.2f})", color="#1a73e8")
ax.axhline(0.95, color="#d93025", ls="--", label="target 95%")
ax.set_xticks(xs); ax.set_xticklabels(NAMES); ax.set_ylim(0, 1.1); ax.set_ylabel("recall (development set)"); ax.legend(loc="lower left"); ax.grid(alpha=0.3, axis="y")
ax.set_title("Figure 3.2 - One targeted change, measured", fontweight="bold")
plt.show()

### 🎛️ Interactive: tune the margin on the development set (never on the test set)

In [ ]:
def margin_demo(margin=18, scene=0):
    r = evaluate(lambda im: pipeline_v2(im, margin), dev)
    fig, axes = plt.subplots(1, 2, figsize=(18, 5.5), gridspec_kw=dict(width_ratios=[1.4, 1]))
    axes[0].imshow(draw(dev[scene][0], lambda im: pipeline_v2(im, margin))[..., ::-1]); axes[0].axis("off")
    axes[0].set_title(f"development scene {scene}")
    axes[1].bar(NAMES, r["recall"], color="#1a73e8"); axes[1].axhline(0.95, color="#d93025", ls="--"); axes[1].set_ylim(0, 1.05)
    axes[1].set_title(f"margin {margin}: recall per class, count error {r['mean_count_error']:.2f}")
    plt.show()


interact(margin_demo, margin=widgets.IntSlider(value=18, min=2, max=80, step=2), scene=widgets.IntSlider(value=0, min=0, max=N_DEV - 1))

### 3.5 Final test, latency and robustness
Only now touch the test sets — once. Report **accuracy**, **speed** (median and 95th-percentile latency, not the best run) and **robustness** (the
dark camera).

In [ ]:
def latency_ms(fn, x, warmup=3, runs=30):
    for _ in range(warmup):
        fn(x)
    ts = []
    for _ in range(runs):
        t0 = time.perf_counter(); fn(x); ts.append((time.perf_counter() - t0) * 1000)
    return float(np.median(ts)), float(np.percentile(ts, 95))


rows = []
for name, pipe in [("v1 Otsu", pipeline_v1), ("v2 belt-relative", pipeline_v2)]:
    r_t, r_d = evaluate(pipe, test), evaluate(pipe, test_dark)
    med, p95 = latency_ms(pipe, test[0][0], runs=5 if SMOKE else 30)
    rows.append((name, r_t["recall"].min(), r_t["precision"].min(), r_t["mean_count_error"], r_d["recall"].min(), med, p95))
table = "| pipeline | worst-class recall | worst-class precision | count error / image | worst-class recall, dark camera | latency median / p95 (ms) |\n|---|---|---|---|---|---|\n"
table += "".join(f"| {n} | {a:.2f} | {b:.2f} | {c:.2f} | {d:.2f} | {e:.1f} / {f:.1f} |\n" for n, a, b, c, d, e, f in rows)
from IPython.display import Markdown, display
display(Markdown("**Table 3.1 - Final results on the untouched test sets**\n\n" + table))

**Read the table like a reviewer.** v2 fixed the bolts, but did it meet the target for *every* class? Look back at Figure 3.2: washers and
flanges stay below 95 % because **touching or overlapping parts merge into one blob** — a limit of the method, not of a parameter. What happens with
the dark camera — and would a fixed margin survive a new belt colour? This is exactly where your project continues: better **lighting** (Week 2) is often the cheapest fix, then a learned
detector trained on your own data (Week 9B), auto-labelled (Week 13A) and augmented or supplemented with synthetic data (Week 13B).

## 4. Evaluation checklist
- [ ] The metric matches the task (per object? per image? per frame? a measurement error in mm?) and the target was fixed **in advance**.
- [ ] Train / validation / test are split **by situation** (day, session, part batch, person, video) — no near-duplicates across splits.
- [ ] Results are **per class** (and the worst class is reported), not only an average.
- [ ] A **baseline** is reported next to your method.
- [ ] Hyper-parameters were tuned on validation data; the test set was used **once**.
- [ ] **Latency** is measured on the demo hardware (median and p95, after warm-up), including pre- and post-processing.
- [ ] **Robustness**: at least one controlled change (lighting, distance, background, camera) is tested.
- [ ] An **error gallery** shows typical failures, with a short explanation of each.
- [ ] Numbers have a sense of uncertainty (how many test objects? would one more error change the conclusion?).

## 5. Templates

### Proposal (Week 5, one page)
1. **Problem statement** (one sentence, with a measurable target).
2. **Why it matters** (who would use it; what does an error cost?).
3. **Set-up**: camera, lens, lighting, working distance; sketch it.
4. **Data plan**: how many images, how they are labelled, how they are split, licences and consent.
5. **Baseline** and **planned method(s)**, linked to course weeks.
6. **Evaluation**: metric, target, test conditions.
7. **Risks and fall-backs** (e.g. "if the detector is too slow, run it every third frame and track in between").
8. **Plan and roles** per group member.

### Final report (short, 4–6 pages + appendix)
Abstract · problem and target · set-up (photo!) · data (table of counts per split and class) · method (a pipeline figure) · results (a results table
like Table 3.1, a confusion matrix, latency) · error analysis (gallery) · limitations and ethics · conclusion · individual contributions · references.

### Demonstration (Week 14)
- **Live** first, slides second: show the system working on objects the audience hands you.
- Show the **numbers** (the results table) and one **failure** you understand.
- Have a **fallback video** in case the live set-up fails.
- Be ready to answer: *what would you do with two more weeks?*

## 6. Reproducibility, code and ethics
- One repository per group with a `README` (how to install, where the data are, how to run the demo), a `requirements.txt`, and fixed random seeds.
- Keep **large files** (datasets, weights, videos) out of git; share them with a link and a checksum.
- Record the **licence** of every dataset, model and library you use (Ultralytics YOLO is AGPL-3.0 — see Week 8B; many foundation models are
  non-commercial).
- **People in images**: consent for filming, blur faces you do not need, store data securely, delete it when the project ends.
- Use AI assistants as you would a colleague: allowed where your instructor says so, **declared** in the report, and you must understand and be
  able to explain every line you submit.

## 7. Exercises (with self-checks)
Replace each `raise NotImplementedError`, then run the check below it.

### E1. Precision, recall and F1 from counts
Given per-class arrays `tp`, `fp`, `fn`, return `(precision, recall, f1)` arrays. Where a denominator is 0, return 0 for that class (no division warnings).

In [ ]:
def prf(tp, fp, fn):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e1():
    p, r, f = prf(np.array([8, 0, 5]), np.array([2, 0, 0]), np.array([0, 0, 5]))
    assert np.allclose(p, [0.8, 0, 1.0]) and np.allclose(r, [1.0, 0, 0.5]), "precision = tp/(tp+fp), recall = tp/(tp+fn)"
    assert np.allclose(f, [2 * 0.8 / 1.8, 0, 2 * 0.5 / 1.5]), "F1 is the harmonic mean of precision and recall"


check("E1 prf", _test_e1)

### E2. How many test objects do you need?
If the true recall is `p` and you test `n` objects, the standard error of the measured recall is $\sqrt{p(1-p)/n}$. Return the **smallest** `n` for which the ±2-standard-error band is at most `half_width` wide on each side.

In [ ]:
def objects_needed(p, half_width):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e2():
    assert objects_needed(0.95, 0.02) == 475, "2*sqrt(0.95*0.05/n) <= 0.02 -> n >= 475"
    assert objects_needed(0.5, 0.1) == 100
    n = objects_needed(0.9, 0.05)
    assert 2 * np.sqrt(0.9 * 0.1 / n) <= 0.05 < 2 * np.sqrt(0.9 * 0.1 / (n - 1)), "return the smallest such n"


check("E2 objects_needed", _test_e2)

### E3. Make the template yours (no self-check)
Replace the `partsim` scenes in Section 3 with 20 photos from your own project, labelled with any tool (CVAT, Label Studio, Roboflow), and run the
same structure: baseline → error gallery → one improvement → results table. This becomes the core of your Week 9 milestone.

In [ ]:
# E3: your code here

## 8. Quick check (auto-marked)

In [ ]:
quiz([
    {
        "q": 'When should the success target of the project be fixed?',
        "options": [
            'After seeing the test results',
            'Before building the system',
            'Never',
            'During the demo',
        ],
        "answer": '32f5533351',
        "explain": 'Otherwise the target is chosen to fit the result.',
    },
    {
        "q": 'Your video frames are split randomly into train and test. What is the risk?',
        "options": [
            'None',
            'Near-identical neighbouring frames leak into the test set',
            'Too few classes',
            'The model trains too slowly',
        ],
        "answer": 'b1797f7a47',
        "explain": 'Split by video, day or session.',
    },
    {
        "q": 'Why report the worst-class recall and not only the mean?',
        "options": [
            'It is smaller',
            'A system can look good on average while failing one important class',
            'It is required by Python',
            'It is faster to compute',
        ],
        "answer": '246b57d988',
        "explain": 'Rare classes hide inside averages.',
    },
    {
        "q": 'Which latency figure is most useful for a live system?',
        "options": [
            'The fastest run',
            'Median and 95th percentile after warm-up, on the demo hardware',
            'The first run',
            "The GPU's theoretical FLOPs",
        ],
        "answer": '40e588998b',
        "explain": 'Include pre- and post-processing.',
    },
    {
        "q": 'The baseline misses dark objects. What is the best next step?',
        "options": [
            'Train a bigger network immediately',
            'Look at the failures and fix the cause (threshold, lighting, data)',
            'Tune on the test set',
            'Remove dark objects from the test set',
        ],
        "answer": '85618a8303',
        "explain": 'Error analysis drives improvement.',
    },
], title='Week 14 project quiz')

## 9. Reflection
1. Write your project's one-sentence problem statement with a measurable target.
2. What will your test set contain that your training set does not (a new day, a new person, new lighting)?
3. What is the cheapest change that could fix your biggest failure group?

## 10. References and further exploration
- Sculley, D. et al. (2015). *Hidden technical debt in machine learning systems.* NeurIPS. [papers.nips.cc](https://papers.nips.cc/paper/5656-hidden-technical-debt-in-machine-learning-systems)
- Kapoor, S. & Narayanan, A. (2023). *Leakage and the reproducibility crisis in machine-learning-based science.* Patterns 4(9). [doi:10.1016/j.patter.2023.100804](https://doi.org/10.1016/j.patter.2023.100804)
- Mitchell, M. et al. (2019). *Model cards for model reporting.* FAT*. [arXiv:1810.03993](https://arxiv.org/abs/1810.03993)
- Gebru, T. et al. (2021). *Datasheets for datasets.* CACM 64(12). [arXiv:1803.09010](https://arxiv.org/abs/1803.09010)
- Ng, A. *A chat with Andrew on MLOps: from model-centric to data-centric AI* (video). [youtube.com/watch?v=06-AZXmwHjo](https://www.youtube.com/watch?v=06-AZXmwHjo)
- Steger, C., Ulrich, M. & Wiedemann, C. (2018). *Machine Vision Algorithms and Applications*, 2nd ed. Wiley — chapters on system design and lighting.
- Labelling tools: [CVAT](https://www.cvat.ai/) · [Label Studio](https://labelstud.io/) · [Roboflow](https://roboflow.com/)

## 11. Key takeaways
- A good project has a **specific, measurable** problem and a target fixed in advance.
- Split data **by situation**, build a **baseline**, look at the **errors**, change **one thing** at a time, and keep a results table.
- Report **per-class** results, **latency** on the demo hardware and **robustness** to a controlled change; touch the test set once.
- Demonstrate live, show one failure you understand, and document data, licences and consent.

Good luck — we look forward to your demonstrations!